# Vesti-Care — Train & Compare 3 Models (Colab GPU) — v2, leakage-fixed

**What changed vs the original `train_models_colab.ipynb` (2026-09 paper audit):**

1. **Genuine 3-way split.** Data now comes from `build_subject_independent_dataset.py`
   (`X_train/val/test.npz`), which assigns *whole subjects* to train/val/test
   independently within each source dataset (SisFall, KFall, VestibularDataset),
   then merges. The old pipeline appended KFall into TRAIN only, so the reported
   TEST set never contained a single KFall window.
2. **No more test-set leakage into training.** `model.fit(..., validation_data=...)`
   now uses the held-out **VAL** split, never test. The old notebook passed
   `(X_test_n, y_te_cat)` as `validation_data`, meaning early stopping picked
   weights based on test-set loss.
3. **Threshold selection moved to VAL.** The CNN+BiLSTM+Attention "production"
   threshold sweep now runs on VAL predictions only, is fully recorded (every
   grid point, not just the winner), and is locked *before* touching TEST. The
   old pipeline's threshold sweep script ran directly on the test set, and the
   headline "T=0.55" production numbers were never saved to any file at all —
   they existed only as a hardcoded string in a plotting script and could not
   be independently re-verified.
4. **One comprehensive JSON per model**, written to Drive, from which every
   paper table/figure should be regenerated — never hand-typed. This is also
   what fixes the T=0.45-vs-T=0.55 mismatch between the old Table 2 and the old
   Figure 7: they were generated by two different, silently-diverged code paths.

Run cells top to bottom. Upload `processed_3class_v2/` (from
`build_subject_independent_dataset.py`) to
`My Drive/VestiCare/data_v2/` before running.

In [ ]:
# Cell 1 — Mount Drive + kiem tra GPU
from google.colab import drive
drive.mount('/content/drive')

import subprocess
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu.returncode == 0:
    print(f'GPU: {gpu.stdout.strip()}')
else:
    print('!!! NO GPU -- Runtime -> Change runtime type -> T4 GPU !!!')


In [ ]:
# Cell 2 -- Duong dan (chinh neu can)
import os

DRIVE_DATA_DIR    = '/content/drive/MyDrive/VestiCare/data_v2'
DRIVE_MODEL_DIR   = '/content/drive/MyDrive/VestiCare/models_v2'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/VestiCare/results_v2'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print('Kiem tra data files:')
for f in ['X_train.npz', 'y_train.npz', 'X_val.npz', 'y_val.npz',
          'X_test.npz', 'y_test.npz', 'origin_test.npz', 'dataset_manifest.json']:
    p = f'{DRIVE_DATA_DIR}/{f}'
    if os.path.exists(p):
        print(f'  [OK] {f}  ({os.path.getsize(p)/1024/1024:.2f} MB)')
    else:
        print(f'  [MISSING] {f} -- run build_subject_independent_dataset.py '
              f'locally and upload processed_3class_v2/ to this Drive path first!')


In [ ]:
# Cell 3 -- Hyperparameters
FOCAL_GAMMA   = 2.0
LR_INIT       = 1e-3
LR_MIN        = 1e-6
RECALL_TARGET = 0.85

CLASS_NAMES  = ['NORMAL', 'INSTABILITY', 'FALL']
NUM_CLASSES  = 3
WINDOW_SIZE  = 100
NUM_CHANNELS = 7

# Fixed a-priori decision thresholds used for the *fair, uniform* baseline
# comparison table across all models (this is not tuned against any split --
# 0.50 is the natural one-vs-rest softmax boundary; kept identical to the
# original pipeline's convention so the comparison table stays comparable).
DEFAULT_T_INST = 0.50
DEFAULT_T_FALL = 0.45

# Grid for the CNN+BiLSTM+Attention *production* threshold sweep, run on VAL
# only. Matches the paper's stated intended range.
SWEEP_GRID = [round(0.30 + 0.02 * i, 2) for i in range(26)]  # 0.30 .. 0.80 step 0.02

# Selection rule for the production operating point, applied to the VAL sweep:
# the pair with the smallest T_FALL that still reaches RECALL_TARGET on FALL
# recall (ties broken by higher FALL precision); if no pair reaches the
# target, fall back to the pair with the single highest FALL recall on VAL.
# State this rule verbatim in the paper's Methods section.
SELECTION_RULE = (
    f"Among all (T_INST, T_FALL) pairs swept on VAL, choose the one with the "
    f"smallest T_FALL such that FALL recall >= {RECALL_TARGET} on VAL, "
    f"breaking ties by higher FALL precision on VAL. If no pair reaches the "
    f"target, choose the pair with the highest FALL recall on VAL."
)

CFG = {
    'lstm': {
        'epochs': 60, 'batch': 128, 'patience': 12, 'fall_w': 6.0,
        'save_as': 'fall_detection_3class_lstm.keras',
    },
    'bilstm': {
        'epochs': 80, 'batch': 128, 'patience': 15, 'fall_w': 6.0,
        'save_as': 'fall_detection_3class_bilstm.keras',
    },
    'cnn_bilstm_attn': {
        'epochs': 80, 'batch': 128, 'patience': 15, 'fall_w': 8.0,
        'save_as': 'fall_detection_3class_cnn_bilstm_attn.keras',
    },
}
print('Config OK')
print(SELECTION_RULE)


In [ ]:
# Cell 4 -- Import + Load data (3-way split)
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print(f'TF {tf.__version__}  |  GPU: {[g.name for g in tf.config.list_physical_devices("GPU")]}')

def _load(name):
    return np.load(f'{DRIVE_DATA_DIR}/{name}.npz')['data']

X_train = _load('X_train').astype(np.float32)
y_train = _load('y_train').astype(np.int32)
X_val   = _load('X_val').astype(np.float32)
y_val   = _load('y_val').astype(np.int32)
X_test  = _load('X_test').astype(np.float32)
y_test  = _load('y_test').astype(np.int32)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
for i, n in enumerate(CLASS_NAMES):
    print(f'  {n:25s}: train={int((y_train==i).sum()):6d}  '
          f'val={int((y_val==i).sum()):5d}  test={int((y_test==i).sum()):5d}')

import json as _json
with open(f'{DRIVE_DATA_DIR}/dataset_manifest.json') as f:
    DATASET_MANIFEST = _json.load(f)
print('\nLoaded dataset_manifest.json (split method, subject lists, provenance).')


In [ ]:
# Cell 5 -- Scaler (tinh tu TRAIN) + Augment (TRAIN only) + Normalize (ca 3 split)

def augment_minority(X, y, seed=42):
    rng = np.random.default_rng(seed)
    n0  = int((y == 0).sum())
    aug_X, aug_y = [X], [y]
    for cls in [1, 2]:
        Xc = X[y == cls]
        if len(Xc) == 0:
            continue
        rep = max(1, n0 // len(Xc) - 1)
        print(f'  {CLASS_NAMES[cls]}: {len(Xc)} windows -> x{rep+1}')
        for _ in range(rep):
            Xn = Xc.copy()
            ch = rng.integers(0, 3, size=len(Xn))
            if (ch==0).any():
                Xn[ch==0] += rng.normal(0, 0.05, Xn[ch==0].shape)
            if (ch==1).any():
                Xn[ch==1] *= rng.uniform(0.85, 1.15, (int((ch==1).sum()), 1, 1))
            if (ch==2).any():
                Xn[ch==2] = np.roll(Xn[ch==2], int(rng.integers(-3, 4)), axis=1)
            aug_X.append(Xn)
            aug_y.append(np.full(len(Xn), cls, dtype=np.int32))
    Xa = np.concatenate(aug_X)
    ya = np.concatenate(aug_y)
    p  = rng.permutation(len(Xa))
    return Xa[p], ya[p]

# Scaler from RAW X_train only -- never from val or test.
MEAN = X_train.mean(axis=(0, 1))
STD  = X_train.std(axis=(0, 1));  STD[STD == 0] = 1.0
print('Scaler (raw X_train):')
print(f'  mean: {np.round(MEAN, 4)}')
print(f'  std : {np.round(STD,  4)}')

SCALER_PATH = f'{DRIVE_MODEL_DIR}/scaler_params.npz'
np.savez(SCALER_PATH, mean=MEAN, std=STD)
print(f'  Saved: {SCALER_PATH}')

print('\nAugmenting TRAIN only...')
X_aug, y_aug = augment_minority(X_train, y_train)
print(f'After augment: {len(X_aug):,} windows')

X_tr_n  = (X_aug  - MEAN) / STD
X_val_n = (X_val  - MEAN) / STD
X_te_n  = (X_test - MEAN) / STD

y_tr_cat  = tf.keras.utils.to_categorical(y_aug,  NUM_CLASSES)
y_val_cat = tf.keras.utils.to_categorical(y_val,  NUM_CLASSES)
y_te_cat  = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)
print('Done.')


In [ ]:
# Cell 6 -- Utilities (focal loss, train-on-VAL, threshold sweep on VAL, eval on TEST)
import time
from tensorflow.keras import layers, models, regularizers

def focal_loss(gamma=2.0):
    def fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        p_t    = tf.reduce_sum(y_true * y_pred, axis=-1)
        return tf.reduce_mean(((1 - p_t) ** gamma) * (-tf.math.log(p_t)))
    fn.__name__ = f'focal_g{gamma}'
    return fn

def make_cw(y, fall_w):
    cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y)
    cw[2] *= fall_w
    return {i: float(v) for i, v in enumerate(cw)}

def run_train(model, cfg, class_weight):
    """Train with VAL as the early-stopping signal. TEST is never passed here."""
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_INIT),
        loss=focal_loss(FOCAL_GAMMA),
        metrics=['accuracy'],
    )
    cbs = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=cfg['patience'],
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=cfg['patience'] // 2, min_lr=LR_MIN, verbose=1,
        ),
    ]
    t0 = time.time()
    h  = model.fit(
        X_tr_n, y_tr_cat,
        epochs=cfg['epochs'], batch_size=cfg['batch'],
        validation_data=(X_val_n, y_val_cat),   # <-- VAL, never test
        class_weight=class_weight,
        callbacks=cbs, verbose=1,
    )
    print(f'Done in {(time.time()-t0)/60:.1f} min | best val_loss={min(h.history["val_loss"]):.4f}')
    return model, h


def _metrics_at_thresholds(y_true, probs, t_inst, t_fall):
    yp = np.zeros(len(y_true), np.int32)
    yp[probs[:, 1] >= t_inst] = 1
    yp[probs[:, 2] >= t_fall] = 2

    cm = confusion_matrix(y_true, yp, labels=[0, 1, 2])
    ftp, ffn, ffp = float(cm[2,2]), float(cm[2,0]+cm[2,1]), float(cm[0,2]+cm[1,2])
    itp, ifn, ifp = float(cm[1,1]), float(cm[1,0]+cm[1,2]), float(cm[0,1]+cm[2,1])
    rec_fall  = ftp/(ftp+ffn) if (ftp+ffn) > 0 else 0.0
    prec_fall = ftp/(ftp+ffp) if (ftp+ffp) > 0 else 0.0
    f1_fall   = 2*rec_fall*prec_fall/(rec_fall+prec_fall) if (rec_fall+prec_fall) > 0 else 0.0
    rec_inst  = itp/(itp+ifn) if (itp+ifn) > 0 else 0.0
    prec_inst = itp/(itp+ifp) if (itp+ifp) > 0 else 0.0
    acc  = float(np.trace(cm)) / float(cm.sum())
    macro_f1 = float(np.mean([
        2*p*r/(p+r) if (p+r) > 0 else 0.0
        for p, r in [
            (cm[i,i]/cm[:,i].sum() if cm[:,i].sum()>0 else 0.0,
             cm[i,i]/cm[i,:].sum() if cm[i,:].sum()>0 else 0.0)
            for i in range(3)
        ]
    ]))
    return dict(
        t_inst=t_inst, t_fall=t_fall,
        recall_fall=round(rec_fall,4), precision_fall=round(prec_fall,4), f1_fall=round(f1_fall,4),
        recall_inst=round(rec_inst,4), precision_inst=round(prec_inst,4),
        accuracy=round(acc,4), macro_f1=round(macro_f1,4),
        confusion_matrix=cm.tolist(),
    )


def sweep_thresholds_on_val(model):
    """Full (T_INST, T_FALL) grid on VAL predictions only. Never touches test."""
    probs_val = model.predict(X_val_n, batch_size=512, verbose=0)
    sweep = []
    for t_inst in SWEEP_GRID:
        for t_fall in SWEEP_GRID:
            sweep.append(_metrics_at_thresholds(y_val, probs_val, t_inst, t_fall))
    return sweep


def select_production_thresholds(sweep):
    qualifying = [row for row in sweep if row['recall_fall'] >= RECALL_TARGET]
    pool = qualifying if qualifying else sweep
    if qualifying:
        pool_sorted = sorted(pool, key=lambda r: (r['t_fall'], -r['precision_fall']))
    else:
        pool_sorted = sorted(pool, key=lambda r: -r['recall_fall'])
    return pool_sorted[0]


def evaluate_on_test(model, t_inst, t_fall):
    t0 = time.perf_counter()
    probs_test = model.predict(X_te_n, batch_size=512, verbose=0)
    ms = (time.perf_counter() - t0) * 1000 / len(y_test)
    result = _metrics_at_thresholds(y_test, probs_test, t_inst, t_fall)
    result['inference_ms_per_window'] = round(ms, 4)
    return result


def evaluate_default_on_test(model, name):
    """Fixed a-priori thresholds, applied uniformly -- the fair-comparison row."""
    result = evaluate_on_test(model, DEFAULT_T_INST, DEFAULT_T_FALL)
    result['name'] = name
    tag = 'PASS' if result['recall_fall'] >= RECALL_TARGET else f"BELOW ({result['recall_fall']:.4f} < {RECALL_TARGET})"
    print(f"  [{name}] FALL recall={result['recall_fall']:.4f}  prec={result['precision_fall']:.4f}  "
          f"F1={result['f1_fall']:.4f}  INST recall={result['recall_inst']:.4f}  [{tag}]")
    return result


def save_model(model, filename):
    path = f'{DRIVE_MODEL_DIR}/{filename}'
    model.save(path)
    print(f'  Saved: {filename}  ({os.path.getsize(path)/1024:.0f} KB)')
    return path


def save_full_results(name, history, default_test_result, extra=None):
    """One comprehensive JSON per model -- the single source every paper
    table/figure must be regenerated from."""
    payload = {
        'model': name,
        'default_thresholds': {'t_inst': DEFAULT_T_INST, 't_fall': DEFAULT_T_FALL},
        'default_test_metrics': default_test_result,
        'training_history': {k: [float(v) for v in vs] for k, vs in history.history.items()},
        'dataset_manifest_split_method': DATASET_MANIFEST.get('split_method'),
        'dataset_merged_totals': DATASET_MANIFEST.get('merged_totals'),
    }
    if extra:
        payload.update(extra)
    out_path = f'{DRIVE_RESULTS_DIR}/{name}_final_metrics.json'
    with open(out_path, 'w') as f:
        json.dump(payload, f, indent=2)
    print(f'  Saved full results: {out_path}')
    return out_path


import json
all_results = []
print('Utilities ready.')


---

In [ ]:
# Cell 7 -- Train LSTM
cfg = CFG['lstm']

inp = layers.Input(shape=(WINDOW_SIZE, NUM_CHANNELS))
x   = layers.LSTM(64, return_sequences=True,
                  kernel_regularizer=regularizers.l2(1e-4))(inp)
x   = layers.Dropout(0.3)(x)
x   = layers.LSTM(32, kernel_regularizer=regularizers.l2(1e-4))(x)
x   = layers.Dropout(0.2)(x)
x   = layers.Dense(32, activation='relu')(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
lstm_model = models.Model(inp, out, name='lstm')
lstm_model.summary()

cw = make_cw(y_aug, cfg['fall_w'])
print(f'Class weights: {cw}')

print('\n--- Training LSTM (validation = VAL split) ---')
lstm_model, lstm_history = run_train(lstm_model, cfg, cw)

print('\n--- Evaluate on TEST (fixed default thresholds) ---')
res = evaluate_default_on_test(lstm_model, 'LSTM')
all_results.append(res)
save_model(lstm_model, cfg['save_as'])
save_full_results('lstm', lstm_history, res)


---

In [ ]:
# Cell 8 -- Train BiLSTM
cfg = CFG['bilstm']

inp = layers.Input(shape=(WINDOW_SIZE, NUM_CHANNELS))
x   = layers.Bidirectional(
          layers.LSTM(64, return_sequences=True,
                      kernel_regularizer=regularizers.l2(1e-4))
      )(inp)
x   = layers.Dropout(0.3)(x)
x   = layers.Bidirectional(
          layers.LSTM(32, kernel_regularizer=regularizers.l2(1e-4))
      )(x)
x   = layers.Dropout(0.2)(x)
x   = layers.Dense(64, activation='relu')(x)
x   = layers.Dropout(0.2)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
bilstm_model = models.Model(inp, out, name='bilstm')
bilstm_model.summary()

cw = make_cw(y_aug, cfg['fall_w'])
print(f'Class weights: {cw}')

print('\n--- Training BiLSTM (validation = VAL split) ---')
bilstm_model, bilstm_history = run_train(bilstm_model, cfg, cw)

print('\n--- Evaluate on TEST (fixed default thresholds) ---')
res = evaluate_default_on_test(bilstm_model, 'BiLSTM')
all_results.append(res)
save_model(bilstm_model, cfg['save_as'])
save_full_results('bilstm', bilstm_history, res)


---

In [ ]:
# Cell 9 -- Train CNN + BiLSTM + Attention (the paper's main / "production" model)
cfg = CFG['cnn_bilstm_attn']

inp = layers.Input(shape=(WINDOW_SIZE, NUM_CHANNELS))

x = layers.Conv1D(64, kernel_size=5, padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(1e-4))(inp)
x = layers.BatchNormalization()(x)
x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True,
                    kernel_regularizer=regularizers.l2(1e-4))
    )(x)
x = layers.Dropout(0.2)(x)

attn = layers.MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.1)(x, x)
x    = layers.Add()([x, attn])
x    = layers.LayerNormalization()(x)

x   = layers.GlobalAveragePooling1D()(x)
x   = layers.Dense(64, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

main_model = models.Model(inp, out, name='cnn_bilstm_attn')
main_model.summary()

cw = make_cw(y_aug, cfg['fall_w'])
print(f'Class weights: {cw}  (FALL_W={cfg["fall_w"]})')

print('\n--- Training CNN+BiLSTM+Attention (validation = VAL split) ---')
main_model, main_history = run_train(main_model, cfg, cw)

print('\n--- Fair-comparison evaluation on TEST (fixed default thresholds) ---')
default_res = evaluate_default_on_test(main_model, 'CNN+BiLSTM+Attn')
all_results.append(default_res)

print('\n--- Threshold sweep on VAL (production operating point) ---')
sweep = sweep_thresholds_on_val(main_model)
production = select_production_thresholds(sweep)
print(f"  Selected: T_INST={production['t_inst']}  T_FALL={production['t_fall']}  "
      f"(VAL recall_fall={production['recall_fall']}, VAL precision_fall={production['precision_fall']})")

print('\n--- Locked-threshold evaluation on TEST (production numbers) ---')
production_test = evaluate_on_test(main_model, production['t_inst'], production['t_fall'])
print(f"  TEST  recall_fall={production_test['recall_fall']}  "
      f"precision_fall={production_test['precision_fall']}  f1_fall={production_test['f1_fall']}  "
      f"accuracy={production_test['accuracy']}")
print('  This -- not any earlier hand-typed number -- is the paper\'s production T value.')

save_model(main_model, cfg['save_as'])
save_full_results(
    'cnn_bilstm_attn', main_history, default_res,
    extra={
        'selection_rule': SELECTION_RULE,
        'threshold_sweep_val': sweep,
        'production_thresholds': {'t_inst': production['t_inst'], 't_fall': production['t_fall']},
        'production_val_metrics': production,
        'production_test_metrics': production_test,
    },
)


In [ ]:
# Cell 10 -- So sanh tat ca models (fair-comparison table, cung 1 threshold policy)
if all_results:
    hdr = f'{"Model":<30} {"Rec-FALL":>9} {"Prec":>7} {"F1":>7} {"Rec-INST":>9} {"ms":>7}'
    print(f'\n{"="*len(hdr)}')
    print(f'  FAIR COMPARISON  @ T_INST={DEFAULT_T_INST}, T_FALL={DEFAULT_T_FALL}  (fixed, not test-tuned)')
    print(f'{"="*len(hdr)}')
    print(hdr)
    for r in all_results:
        mk = ' <' if r['recall_fall'] >= RECALL_TARGET else ''
        print(f"  {r['name']:<28} {r['recall_fall']:>9.4f} {r['precision_fall']:>7.4f} "
              f"{r['f1_fall']:>7.4f} {r['recall_inst']:>9.4f} {r.get('inference_ms_per_window',0):>7.4f}{mk}")
    print('  < = FALL recall >= 0.85 target')

    result_path = f'{DRIVE_RESULTS_DIR}/baseline_comparison_v2.json'
    with open(result_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'\nSaved: {result_path}')
else:
    print('Khong co ket qua nao -- chay Cell 8, 10, 12 truoc')


---
## Finding-4 check: dataset-origin confounding

INSTABILITY windows come only from VestibularDataset; NORMAL/FALL come from
all three sources. This cell trains a small classifier whose *only* job is to
guess which source dataset a window came from. If it can do that with
suspiciously high accuracy, a reviewer's worry -- that the 3-class model
might partly be keying on acquisition-pipeline artefacts rather than genuine
motion -- gains support and should be discussed, not hidden.

In [ ]:
# Cell 11 -- Dataset-origin confound check (vestibular vs sisfall+kfall)
origin_train = np.load(f'{DRIVE_DATA_DIR}/origin_train.npz')['data']
origin_test  = np.load(f'{DRIVE_DATA_DIR}/origin_test.npz')['data']

# Binary target: is this window from the vestibular pipeline (the sole
# INSTABILITY source) or from SisFall/KFall?
z_train = (origin_train == 'vestibular').astype(np.int32)
z_test  = (origin_test  == 'vestibular').astype(np.int32)
print(f'Train origin balance: vestibular={int(z_train.sum())} / other={int((1-z_train).sum())}')
print(f'Test  origin balance: vestibular={int(z_test.sum())} / other={int((1-z_test).sum())}')

# Reuse the plain (un-augmented) normalized train/test tensors -- this must
# NOT reuse the class-balancing augmentation built for the 3-class task, so
# origin balance stays whatever it naturally is.
X_tr_origin_n = (X_train - MEAN) / STD

inp = layers.Input(shape=(WINDOW_SIZE, NUM_CHANNELS))
x = layers.Conv1D(32, kernel_size=5, padding='same', activation='relu')(inp)
x = layers.BatchNormalization()(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(32, activation='relu')(x)
out = layers.Dense(1, activation='sigmoid')(x)
origin_model = models.Model(inp, out, name='origin_confound_probe')

origin_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                      loss='binary_crossentropy', metrics=['accuracy'])
origin_model.fit(
    X_tr_origin_n, z_train, epochs=20, batch_size=128, validation_split=0.15,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1,
)

origin_probs = origin_model.predict(X_te_n, batch_size=512, verbose=0).ravel()
origin_pred  = (origin_probs >= 0.5).astype(np.int32)
origin_acc   = float((origin_pred == z_test).mean())
print(f'\nDataset-origin classifier TEST accuracy: {origin_acc:.4f}')
print('Interpretation guide: close to 0.5 = sources are hard to tell apart from')
print('raw signal alone (reassuring); close to 1.0 = sources are trivially')
print('separable, so any single-source-driven pattern (e.g. INSTABILITY) is at')
print('elevated risk of partly reflecting acquisition-pipeline identity rather')
print('than genuine motion difference. Report this number in Results regardless')
print('of which way it comes out.')

with open(f'{DRIVE_RESULTS_DIR}/domain_origin_confound_check.json', 'w') as f:
    json.dump({
        'target': 'vestibular vs (sisfall+kfall)',
        'test_accuracy': round(origin_acc, 4),
        'train_balance': {'vestibular': int(z_train.sum()), 'other': int((1-z_train).sum())},
        'test_balance': {'vestibular': int(z_test.sum()), 'other': int((1-z_test).sum())},
    }, f, indent=2)
print(f'Saved: {DRIVE_RESULTS_DIR}/domain_origin_confound_check.json')


---

In [ ]:
# Cell 12 -- Download tat ca ve may
import os
from google.colab import files

def convert_tflite(keras_path):
    tflite_path = keras_path.replace('.keras', '.tflite')
    try:
        import tensorflow as tf
        m    = tf.keras.models.load_model(keras_path, compile=False)
        conv = tf.lite.TFLiteConverter.from_keras_model(m)
        conv.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS,
        ]
        conv._experimental_lower_tensor_list_ops = False
        b = conv.convert()
        with open(tflite_path, 'wb') as f:
            f.write(b)
        print(f'  TFLite: {os.path.basename(tflite_path)}  ({len(b)//1024} KB)')
    except Exception as e:
        print(f'  TFLite failed: {e}')

keras_files = [
    CFG['lstm']['save_as'],
    CFG['bilstm']['save_as'],
    CFG['cnn_bilstm_attn']['save_as'],
]

print('=== Converting TFLite ===')
for fname in keras_files:
    path = f'{DRIVE_MODEL_DIR}/{fname}'
    if os.path.exists(path):
        convert_tflite(path)
    else:
        print(f'  [SKIP] {fname} not on Drive')

print('\n=== Downloading .keras ===')
for fname in keras_files:
    path = f'{DRIVE_MODEL_DIR}/{fname}'
    if os.path.exists(path):
        print(f'  {fname}')
        files.download(path)

print('\n=== Downloading scaler_params.npz ===')
files.download(SCALER_PATH)

print('\n=== Downloading all results_v2/*.json ===')
for fname in os.listdir(DRIVE_RESULTS_DIR):
    if fname.endswith('.json'):
        print(f'  {fname}')
        files.download(f'{DRIVE_RESULTS_DIR}/{fname}')

print('\n[DONE] Copy the .keras files into ml/models_3class/, and copy every')
print('results_v2/*.json into results/ -- every paper table/figure must be')
print('regenerated from these JSON files, never hand-typed.')
